# 文本分类实例

## Step1 导入相关包

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## Step2 加载数据

In [2]:
import pandas as pd

data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


In [3]:
data = data.dropna()
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


## Step3 创建Dataset

In [4]:
from torch.utils.data import Dataset

class MyDataset(Dataset):

    def __init__(self) -> None:
        super().__init__()
        self.data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
        self.data = self.data.dropna()

    def __getitem__(self, index):
        return self.data.iloc[index]["review"], self.data.iloc[index]["label"]
    
    def __len__(self):
        return len(self.data)

In [5]:
dataset = MyDataset()
for i in range(5):
    print(dataset[i])

('距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.', 1)
('商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!', 1)
('早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。', 1)
('宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小，但加上低价位因素，还是无超所值的；环境不错，就在小胡同内，安静整洁，暖气好足-_-||。。。呵还有一大优势就是从宾馆出发，步行不到十分钟就可以到梅兰芳故居等等，京味小胡同，北海距离好近呢。总之，不错。推荐给节约消费的自助游朋友~比较划算，附近特色小吃很多~', 1)
('CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风', 1)


## Step4 划分数据集

In [6]:
from torch.utils.data import random_split


trainset, validset = random_split(dataset, lengths=[0.9, 0.1])
len(trainset), len(validset)

(6989, 776)

In [7]:
for i in range(10):
    print(trainset[i])

('我们四号入住万豪的行政豪华海景房,感觉很好.首先从大堂的感觉就很舒服,之前在网上看到有人评论说办理入住手续慢,对待外国客人要比中国人热情以及忘记送冰毛巾,欢迎项链和饮料的情况都不存在,我们很满意.第一晚的马桶出了些状况,不过服务员很快就把卫生间搞定了,相当满意.泳池1.2米,不过适合我,不会游泳.海滩的沙子很细软,工作人员也很负责,总之到处都能感受到5星服务的标准.所以的服务员也都很热情.虽然不错,不过我们下次要试一下丽滋酒店,呵呵.....没有比较,就没有鉴别嘛!', 1)
('已经住过多次了，还是一个字：好！服务好，环境好！服务人性化，房间虽然小些，但对我足够了。一点儿遗憾是没有免费的矿泉水。早餐不错，很丰盛尤其是有一些小笼，像南方的早茶了。对于3星级有此早餐很好了。携程的价格还是稍差一些，我以其他公司协议价住过要更便宜些。总的讲：下次还住，还推荐其它同事住。', 1)
('以前住过这家饭店，但那时还是HILTON管理的，现在是NIKKO了，感觉明显不如以前，并且设施也有些陈旧了，但在大连五星级饭店，这家饭店的价位相对来说还比较合理，酒店附近有个新建的时尚生活区，有一些好的餐厅和咖啡店不错，酒店综合服务和管理还需要加强。', 1)
('我订的是高级间，因为不吸烟，很幸运的升级到行政楼18楼，房间很大,服务人员很亲善，比九龙酒店（房间小，服务人员没笑容）好太多，感觉很好,下次还会来帝京住的.', 1)
('再也不会住这家店了。首先房间隔音差了要命，隔壁的一举一动清清楚楚，半夜居然还关了空调，这么冷的天，都冻感冒了，实在差劲。', 0)
('这次去香港之前,我们就在携程上找酒店,后来定了朗豪酒店,因为看到对它的评价挺高的,入住后才发现真的很不错!前台和服务员态度都很好,CHECKIN和CHECKOUT的速度也快,整个酒店的装修很时尚,特别是房间卫生间的设计很前卫,非常喜欢!而且酒店地理位置也很好,交通十分方便,酒店直通对面的朗豪坊,下面就可以直接到达地铁站,去哪里都非常方便!酒店附近就是波鞋街,女人街,电器街,逛街也方便,下次去香港还住朗豪酒店!', 1)
('没见过这么差的四星酒店!建议大家以后千万别住,离银滩很远很远!环境也很一般,服务就差得没法提了!', 0)
('很好的一家酒店哦，以后还会去呢！交通方便，服务好，不错不错', 1)
('我本人办的有锦江之

## Step5 创建Dataloader

In [8]:
import torch

tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-roberta-wwm-ext")

def collate_func(batch):
    texts, labels = [], []
    for item in batch:
        texts.append(item[0])
        labels.append(item[1])
    inputs = tokenizer(texts, max_length=128, padding="max_length", truncation=True, return_tensors="pt")
    inputs["labels"] = torch.tensor(labels)
    return inputs

In [22]:
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, collate_fn=collate_func)
validloader = DataLoader(validset, batch_size=64, shuffle=False, collate_fn=collate_func)

In [10]:
next(enumerate(validloader))[1]

{'input_ids': tensor([[  101,  4685,   928,  ...,     0,     0,     0],
        [  101,  3300,  2523,  ...,     0,     0,     0],
        [  101,  6821,  2157,  ...,     0,     0,     0],
        ...,
        [  101,  7370,   749,  ...,     0,     0,     0],
        [  101,  1392,   855,  ...,  8013, 10887,   102],
        [  101,  6983,  2421,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1,
        1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 

## Step6 创建模型及优化器

In [11]:
from torch.optim import Adam

model = AutoModelForSequenceClassification.from_pretrained("hfl/chinese-roberta-wwm-ext")

if torch.cuda.is_available():
    model = model.cuda()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/chinese-roberta-wwm-ext and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
## dp代码
model = torch.nn.DataParallel(model,device_ids=[0,1,2])

model.device_ids

[0, 1, 2]

In [13]:
model  # 已经有DataParallel包装了

DataParallel(
  (module): BertForSequenceClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(21128, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=7

In [ ]:
model.module  # 这才是我们的模型

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

## 数据并行对推理的影响

快乐一丢丢

In [23]:
# 单GPU推理
%time
with torch.inference_mode():
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            output = model.module(**batch)

CPU times: user 37 μs, sys: 2 μs, total: 39 μs
Wall time: 76.3 μs


In [24]:
# 多GPU推理
with torch.inference_mode():
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            output = model(**batch)

In [14]:
optimizer = Adam(model.parameters(), lr=2e-5)

## Step7 训练与验证

In [15]:
import time

def evaluate():
    model.eval()
    acc_num = 0
    with torch.inference_mode():
        for batch in validloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output.logits, dim=-1)
            acc_num += (pred.long() == batch["labels"].long()).float().sum()
    return acc_num / len(validset)

def train(epoch=3, log_step=100):
    global_step = 0
    for ep in range(epoch):
        model.train()
        start = time.time()
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            optimizer.zero_grad()
            output = model(**batch)
            # loss变为标量 
            loss = output.loss.mean()
            loss.backward()
            optimizer.step()
            if global_step % log_step == 0:
                print(f"ep: {ep}, global_step: {global_step}, loss: {loss.mean().item()}")
            global_step += 1
        acc = evaluate()
        print(f"ep: {ep}, acc: {acc}, time: {time.time() - start}")

## Step8 模型训练

In [16]:
train()

/data/ljr/anaconda3/envs/dpspeed/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


ep: 0, global_step: 0, loss: 1.0127347707748413
ep: 0, global_step: 100, loss: 0.287835955619812
ep: 0, global_step: 200, loss: 0.07999441772699356
ep: 0, acc: 0.8994845151901245, time: 23.58604860305786
ep: 1, global_step: 300, loss: 0.26766616106033325
ep: 1, global_step: 400, loss: 0.19874005019664764
ep: 1, acc: 0.894329845905304, time: 22.378119230270386
ep: 2, global_step: 500, loss: 0.016678372398018837
ep: 2, global_step: 600, loss: 0.14647600054740906
ep: 2, acc: 0.8994845151901245, time: 22.570558786392212


单GPU训练56.8s batch 32

只占用了一个卡

 0   N/A  N/A         3581505      C   ...onda3/envs/dpspeed/bin/python       4362MiB




三个GPU dp训练 68s  batch 32/3 *3

显存占用如下
| GPU | GI  | CI    | PID     | Type | Process Name                             | GPU Memory |
|-----|-----|-------|---------|------|------------------------------------------|------------|
| 0   | N/A | N/A   | 3582667 | C    | ...onda3/envs/dpspeed/bin/python         | 3016MiB    |
| 1   | N/A | N/A   | 2849    | G    | /usr/lib/xorg/Xorg                        | 4MiB       |
| 1   | N/A | N/A   | 60601   | G    | /usr/lib/xorg/Xorg                        | 4MiB       |
| 1   | N/A | N/A   | 173850  | G    | /usr/lib/xorg/Xorg                        | 4MiB       |
| 1   | N/A | N/A   | 3582667 | C    | ...onda3/envs/dpspeed/bin/python         | 2006MiB    |
| 2   | N/A | N/A   | 2849    | G    | /usr/lib/xorg/Xorg                        | 4MiB       |
| 2   | N/A | N/A   | 60601   | G    | /usr/lib/xorg/Xorg                        | 4MiB       |
| 2   | N/A | N/A   | 173850  | G    | /usr/lib/xorg/Xorg                        | 4MiB       |
| 2   | N/A | N/A   | 3582667 | C    | ...onda3/envs/dpspeed/bin/python         | 1868MiB    |


这里的dp也一般 效果不大行

## Step9 模型预测

In [36]:
sen = "我觉得这家酒店不错，饭很好吃！"
id2_label = {0: "差评！", 1: "好评！"}
model.eval()
with torch.inference_mode():
    inputs = tokenizer(sen, return_tensors="pt")
    inputs = {k: v.cuda() for k, v in inputs.items()}
    logits = model(**inputs).logits
    pred = torch.argmax(logits, dim=-1)
    print(f"输入：{sen}\n模型预测结果:{id2_label.get(pred.item())}")

输入：我觉得这家酒店不错，饭很好吃！
模型预测结果:好评！


In [37]:
from transformers import pipeline

model.config.id2label = id2_label
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

In [38]:
pipe(sen)

[{'label': '好评！', 'score': 0.9991635084152222}]

In [ ]:
import torch

# 在大模型训练或推理后释放缓存
torch.cuda.empty_cache()